# 02 — Single-hop navigator dry run

This notebook drives **one hop** of the TypeSafe `system_one` navigator in
`src/neo4jev/navigator.py` against the live **Companies KG**:

1. pick a real start node with `neo4j_access.search_start_nodes(...)`
2. fetch its **capped** outgoing relationships with `get_outgoing_relationships(...)`
3. ask TypeSafe **one** `system_one` question set — a `Choice` over the outgoing
   edges keyed `e0`, `e1`, … (with the side table back to `NavCandidate`) plus a
   `Noul` asking "has the goal already been reached?"
4. inspect the returned probability distribution, the Noul value, and the
   `top_k` / `cutoff` selection built on top of it

Exactly **one** real `system_one` call is issued here, so the hop's raw output can
be sanity-checked visually before it disappears inside the beam search of notebook 03.

> **API key requirement.** Step 3 needs `TYPESAFE_API_KEY` in `.env`. Steps 1 and 2 run
> against the live graph without it. Step 4's selection logic is exercised without a key
> on **explicitly synthetic** input (never presented as TypeSafe output); its live form
> needs the same key. If the key is missing the call is still attempted and the failure
> is displayed verbatim.

## Setup — load `.env`, connect to the Companies KG

`neo4j_access.settings_from_env()` reads only the Neo4j variables, so graph access does
not depend on the TypeSafe key.

In [1]:
import json
import os
from contextlib import ExitStack

from dotenv import load_dotenv
from IPython.display import Markdown, display
from typesafe_sdk import AsyncTypeSafeClient, TypeSafeError

from neo4jev import neo4j_access
from neo4jev.navigator import NavigatorConfig, NodeContext, _select_branches, one_hop
from neo4jev.types import FreeTextGoal

load_dotenv()

settings = neo4j_access.settings_from_env()
typesafe_api_key = os.environ.get("TYPESAFE_API_KEY", "").strip()
TYPESAFE_READY = bool(typesafe_api_key)

print(f"Neo4j    : {settings.neo4j_uri} / database {settings.neo4j_database}")
print(f"TypeSafe : {'TYPESAFE_API_KEY found' if TYPESAFE_READY else 'TYPESAFE_API_KEY missing or empty'}")

Neo4j    : neo4j+s://demo.neo4jlabs.com:7687 / database companies2
TypeSafe : TYPESAFE_API_KEY missing or empty


In [2]:
stack = ExitStack()
access = stack.enter_context(neo4j_access.open_access(settings))

labels = access.list_labels()
print(f"connected to {settings.neo4j_database}: {len(labels)} labels")
print(", ".join(labels))

connected to companies2: 15 labels
Article, CPCClass, Chunk, City, Country, IPCClass, IndustryCategory, Investment, NAICSCode, Organization, Patent, Person, Region, SECFiling, Technology


## Step 1 — pick a real start node

`search_start_nodes` uses the label's own indexes, discovered live — here the
`organization_fullName` fulltext index for `Organization`. The graph holds **three**
organizations whose full name begins with "Apple" and all three tie on fulltext score, so
`limit=1` is not deterministic; the start node is instead picked from the top hits by
`(score desc, element_id asc)`, which is stable across runs. On this graph that lands on
`Apple Inc.`

Captions below are a notebook-local convenience (the longest string property that is not
a URL); `viz.py` applies its own caption heuristic when rendering the graph.


In [3]:
def caption(props, fallback):
    strings = [
        value for value in props.values() if isinstance(value, str) and value.strip()
    ]
    readable = [value for value in strings if not value.startswith("http")]
    if readable:
        return max(readable, key=len)[:60]
    return strings[0][:60] if strings else fallback


LOOKUP_LABEL = "Organization"
LOOKUP_QUERY = "Apple"

hits = access.search_start_nodes(LOOKUP_LABEL, LOOKUP_QUERY, "fulltext", limit=5)
if not hits:
    raise RuntimeError(
        f"no {LOOKUP_LABEL} matched {LOOKUP_QUERY!r}: check the query, or "
        f"access.detect_indexes({LOOKUP_LABEL!r}) for the lookup modes this label has"
    )

for hit in hits:
    print(f"{hit.score:8.4f}  {hit.element_id}  {caption(hit.props, hit.label)}")

start_hit = sorted(hits, key=lambda hit: (-(hit.score or 0.0), hit.element_id))[0]
print()
print(f"start node: {caption(start_hit.props, start_hit.label)}  ({start_hit.element_id})")

  4.2230  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:8044  Apple Music is a streaming service offering songs, albums, c
  4.2230  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:8615  APPLE ADS designs, manufactures, and installs signage boards
  4.2230  4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:4  Apple Inc. designs, manufactures, and markets smartphones, p

start node: Apple Inc. designs, manufactures, and markets smartphones, p  (4:c5dede6a-3906-4af7-9f45-70d1fa7d6ade:4)


## Step 2 — outgoing relationships (capped, schema-agnostic)

Apple Inc. is a supernode: **1354** outgoing edges. `get_outgoing_relationships` caps per
relationship type and in total, and reports the shortfall as "N of M edges considered"
so the truncation is visible rather than silent. Relationship types and properties are
read from the live graph — nothing is hardcoded — and edge keys `e0..eN` are assigned
positionally, because several relationships of the same type to different targets would
otherwise collapse into one option.

Read the per-type lines below closely. Relationship types are visited in **name order**, so
the 60-edge total cap is filled by the alphabetically early types (`APPLIED_FOR` …
`HAS_CATEGORY`) while every later one reports "0 of …" — `HAS_COMPETITOR`,
`HAS_PARTNER`, `HAS_SUBSIDIARY`, `HAS_SUPPLIER`, `USES_TECHNOLOGY` and several more.
That is the cap working as specified, and it matters when reading a traversal: on a supernode like this one the model
never gets to choose from the edges further down the alphabet. The cap is a parameter, not
a constant, so raising `total_cap` is how you widen that window — notebook 03 picks its
start node with this in mind.


In [4]:
outgoing = access.get_outgoing_relationships(start_hit.element_id)
by_key = dict(outgoing.mapping)

print(f"capping: {outgoing.note}  (truncated={outgoing.truncated})")
for rel_type in sorted(outgoing.by_type_note):
    print(f"  {outgoing.by_type_note[rel_type]}")

print()
print(f"{'edge_key':<10}{'rel_type':<22}target")
for candidate in outgoing.candidates:
    target = caption(candidate.target_props, candidate.target_label)
    print(f"{candidate.edge_key:<10}{candidate.rel_type:<22}{candidate.target_label}: {target}")

capping: 60 of 1354 edges considered  (truncated=True)
  10 of 25 APPLIED_FOR edges considered
  10 of 25 ASSIGNED edges considered
  10 of 47 CLASSIFIED_AS edges considered
  10 of 79 FILED edges considered
  3 of 3 FOUNDED_BY edges considered
  10 of 17 HAS_BOARD_MEMBER edges considered
  7 of 10 HAS_CATEGORY edges considered
  0 of 1 HAS_CEO edges considered
  0 of 26 HAS_COMPETITOR edges considered
  0 of 26 HAS_CUSTOMER edges considered
  0 of 9 HAS_INVESTMENT edges considered
  0 of 84 HAS_PARTNER edges considered
  0 of 190 HAS_SUBSIDIARY edges considered
  0 of 100 HAS_SUPPLIER edges considered
  0 of 10 INVESTED_IN edges considered
  0 of 21 IN_CITY edges considered
  0 of 681 USES_TECHNOLOGY edges considered

edge_key  rel_type              target
e0        APPLIED_FOR           Patent: Methods, systems, and computer readable storage medium relat
e1        APPLIED_FOR           Patent: Methods, systems, and computer readable storage medium relat
e2        APPLIED_FOR         

## Step 3 — one hop: a single `system_one` call (Choice + Noul)

`one_hop` builds one `Choice` question over the capped edges and one `Noul` question
("has the goal already been reached before following any further relationship?"), sends
them in a **single** `system_one` call, and maps the returned `e*` probabilities back
to `NavCandidate`s through the side table. `top_k` and `cutoff` then select the
branches a beam search would keep.

A node with no outgoing edges still gets its call: the `Choice` is dropped (there is
nothing to choose between) and the `Noul` decides whether the dead end already satisfies
the goal.

In [5]:
goal = FreeTextGoal(goal="move toward an organization that competes with the start organization")
config = NavigatorConfig(top_k=2, cutoff=0.05)

hop = None
typesafe_error = None
try:
    client = AsyncTypeSafeClient(api_key=typesafe_api_key)
    async with client:
        hop = await one_hop(
            client,
            NodeContext(
                element_id=start_hit.element_id,
                label=start_hit.label,
                properties=start_hit.props,
            ),
            list(outgoing),
            goal,
            config=config,
        )
except TypeSafeError as error:
    typesafe_error = error

if hop is None:
    display(
        Markdown(
            "**The one `system_one` call failed — this run has no live TypeSafe output.**\n\n"
            f"`{type(typesafe_error).__name__}: {typesafe_error}`\n\n"
            "Fill `TYPESAFE_API_KEY` in `.env` and re-run this cell to see live output. "
            "The cells below still execute: they display whatever a successful hop returns, "
            "and where the hop is unavailable they exercise the selection logic on input that "
            "is labelled synthetic."
        )
    )
else:
    print(f"one_hop answered at node {hop.node_id}")
    print(f"options offered          : {len(hop.candidates)}")
    print(f"probabilities returned   : {len(hop.probabilities)}")
    print(f"noul (P(goal already reached)) = {hop.noul}")
    print(f"choice confidence = {hop.confidence}")

**The one `system_one` call failed — this run has no live TypeSafe output.**

`TypeSafeAPIConnectionError: Connection error: Illegal header value b'Bearer '`

Fill `TYPESAFE_API_KEY` in `.env` and re-run this cell to see live output. The cells below still execute: they display whatever a successful hop returns, and where the hop is unavailable they exercise the selection logic on input that is labelled synthetic.

## What came back

The `Choice` answer is a probability distribution over the option keys — one entry per edge
the model scored — and the `Noul` answer is a single probability that the goal was already
satisfied at the current node. They are independent: the Noul can be high while the option
distribution is flat (and vice versa). The distribution is what the beam search consumes,
via `sum(log(p))` over the hops of a path.

`HopResult` also carries the SDK's per-answer `ChoiceAnswer.confidence` — how sure the model
was of the option it actually picked. It is displayed here for inspection, but path scoring
deliberately uses the option probabilities alone: confidence is a property of the *answer*,
not of each option, so it has no place in a per-edge `sum(log(p))`.


In [6]:
if hop is not None:
    print(f"full probabilities dict ({len(hop.probabilities)} entries, edge_key -> probability):")
    print(json.dumps(hop.probabilities, indent=2, sort_keys=True))
    print()
    print(f"sum of option probabilities = {sum(hop.probabilities.values()):.6f}")
    print(f"noul = {hop.noul}")
    print(f"choice confidence = {hop.confidence}")
    print()
    hop_mapping = {candidate.edge_key: candidate for candidate in hop.candidates}
    print(f"{'rank':<6}{'edge_key':<10}{'probability':<14}{'rel_type':<22}target")
    ranked = sorted(hop.probabilities.items(), key=lambda item: -item[1])
    for rank, (key, probability) in enumerate(ranked, start=1):
        candidate = hop_mapping.get(key)
        target = caption(candidate.target_props, candidate.target_label) if candidate else "?"
        rel_type = candidate.rel_type if candidate else "?"
        print(f"{rank:<6}{key:<10}{probability:<14.6f}{rel_type:<22}{target}")
else:
    display(Markdown("_Skipped: the hop returned no probabilities (see the note in the previous cell)._"))

_Skipped: the hop returned no probabilities (see the note in the previous cell)._

## Step 4 — top-k / cutoff selection over the returned probabilities

`select_options` below is a readable copy of `navigator._select_branches`, the rule the
beam search applies to every hop: rank the option probabilities, keep those at or above
`cutoff`, and never take more than `top_k` of them. If **nothing** clears the cutoff, the
single best option is kept anyway, so a hop is never dropped purely because of a
threshold.

Because it is a copy, both branches below call the library's own `_select_branches` on
the same inputs and report whether the two agree — the private name is imported
deliberately, to make that equivalence check real rather than a comment.

In [7]:
def select_options(probabilities, mapping, *, top_k, cutoff):
    ranked = sorted(
        ((key, probability) for key, probability in probabilities.items() if key in mapping),
        key=lambda item: item[1],
        reverse=True,
    )
    if not ranked:
        return []
    above_cutoff = [(key, probability) for key, probability in ranked if probability >= cutoff]
    selected = above_cutoff or ranked[:1]
    return [(mapping[key], probability) for key, probability in selected[: max(1, top_k)]]


def same_selection(left, right):
    return [(candidate.edge_key, round(probability, 9)) for candidate, probability in left] == [
        (candidate.edge_key, round(probability, 9)) for candidate, probability in right
    ]

In [8]:
if hop is not None:
    hop_mapping = {candidate.edge_key: candidate for candidate in hop.candidates}

    for top_k in (1, 2, 3):
        branches = select_options(hop.probabilities, hop_mapping, top_k=top_k, cutoff=config.cutoff)
        library = _select_branches(hop.probabilities, hop_mapping, top_k=top_k, cutoff=config.cutoff)
        print(f"top_k={top_k}, cutoff={config.cutoff} -> {len(branches)} branch(es)")
        for candidate, probability in branches:
            print(
                f"    {candidate.edge_key:<5}{probability:.6f}  {candidate.rel_type:<22}"
                f"{caption(candidate.target_props, candidate.target_label)}"
            )
        print(f"    matches navigator._select_branches: {same_selection(branches, library)}")

    print()
    print("one_hop's own selection for the same hop:")
    for candidate, probability in hop.chosen:
        print(f"    {candidate.edge_key:<5}{probability:.6f}  {candidate.rel_type}")
    print()
    print(f"local selection matches one_hop.chosen: {same_selection(hop.chosen, select_options(hop.probabilities, hop_mapping, top_k=config.top_k, cutoff=config.cutoff))}")

    print()
    for cutoff in (0.0, 0.1, 0.25, 0.5):
        kept = [
            key
            for key, probability in sorted(hop.probabilities.items(), key=lambda item: -item[1])
            if probability >= cutoff
        ]
        print(f"cutoff={cutoff:<5}keeps {len(kept)} of {len(hop.probabilities)} options")
else:
    display(Markdown("_Skipped: no live probabilities to select from._"))

_Skipped: no live probabilities to select from._

### Synthetic input check (runs with or without an API key)

The same `select_options` function is run over a **uniform distribution** built from the
real `e0..eN` keys of this hop, so the mechanics stay verifiable without TypeSafe. This is
**not** TypeSafe output — it is a fixed input that shows the `cutoff` rule and its
keep-the-best fallback. With `top_k=3, cutoff=0.5` every probability equals `1/60 ≈ 0.0167`,
so nothing clears the cutoff and the fallback returns exactly one branch.

In [9]:
count = len(outgoing.candidates)
if count == 0:
    display(
        Markdown(
            "_Skipped: the start node has no outgoing edges, so there is no candidate "
            "set to build a synthetic distribution from._"
        )
    )
else:
    sample = {candidate.edge_key: 1.0 / count for candidate in outgoing.candidates}
    print(f"synthetic uniform distribution over {count} candidates (not TypeSafe output)")
    print()
    for top_k, cutoff in ((3, 0.0), (3, 0.5)):
        branches = select_options(sample, by_key, top_k=top_k, cutoff=cutoff)
        library = _select_branches(sample, by_key, top_k=top_k, cutoff=cutoff)
        print(
            f"top_k={top_k}, cutoff={cutoff} -> {[candidate.edge_key for candidate, _ in branches]}"
            f"  (matches navigator._select_branches: {same_selection(branches, library)})"
        )

synthetic uniform distribution over 60 candidates (not TypeSafe output)

top_k=3, cutoff=0.0 -> ['e0', 'e1', 'e2']  (matches navigator._select_branches: True)
top_k=3, cutoff=0.5 -> ['e0']  (matches navigator._select_branches: True)


## Takeaways

- One hop = one `system_one` call, carrying both the `Choice` over the capped outgoing
  edges and the `Noul` goal check.
- Option probabilities arrive keyed by opaque `e*` ids and are mapped back to
  `NavCandidate`s through a side table, so two edges of the same type stay distinct options.
- `top_k` / `cutoff` turn that distribution into branches, with a keep-the-best fallback
  so no hop is dropped purely because of the threshold.
- The Noul value is the goal test the beam search uses to stop a branch early; it is
  reported separately from the option distribution and is never folded into it.
- Truncation is explicit: Apple Inc. exposes 1354 outgoing edges, of which 60 reach the model.
  Anything the cap leaves out cannot be chosen, which is the price of bounded prompts.

In [10]:
stack.close()
print("Neo4j driver closed")

Neo4j driver closed
